# GradeScope — 02. Preprocessing

**Zbiór danych:** Student Performance Factors — 6607 rekordów, 19 cech  
**Cel:** uzupełnienie braków, kodowanie zmiennych kategorycznych, podział train/test

In [7]:
import pandas as pd
import numpy as np
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

df = pd.read_csv("../data/student_performance.csv")
print(f"Wczytano: {df.shape[0]} rekordów, {df.shape[1]} kolumn")

Wczytano: 6607 rekordów, 20 kolumn


## 1. Uzupełnienie brakujących wartości

In [8]:
# Kolumny z brakującymi wartościami
cols_with_missing = ["Teacher_Quality", "Parental_Education_Level", "Distance_from_Home"]

for col in cols_with_missing:
    moda = df[col].mode()[0]
    df[col] = df[col].fillna(moda)
    print(f"{col}: uzupełniono modą → '{moda}'")

print(f"\nBrakujące wartości po uzupełnieniu: {df.isnull().sum().sum()}")

Teacher_Quality: uzupełniono modą → 'Medium'
Parental_Education_Level: uzupełniono modą → 'High School'
Distance_from_Home: uzupełniono modą → 'Near'

Brakujące wartości po uzupełnieniu: 0


## 2. Zmienna docelowa — Pass/Fail

In [9]:
# Binaryzacja Exam_Score — próg 67 (mediana zbioru)
df["Pass"] = (df["Exam_Score"] >= 67).astype(int)  # 1 = Pass, 0 = Fail

print("Rozkład zmiennej docelowej:")
print(df["Pass"].value_counts().rename({1: "Pass (1)", 0: "Fail (0)"}))

# Usuwamy oryginalną kolumnę Exam_Score — nie będzie cechą wejściową
df = df.drop(columns=["Exam_Score"])

Rozkład zmiennej docelowej:
Pass
Pass (1)    3725
Fail (0)    2882
Name: count, dtype: int64


## 3. Kodowanie zmiennych kategorycznych

In [10]:
# LabelEncoder zamiast OneHotEncoder — cechy kategoryczne są porządkowe (Low < Medium < High)
# Kodowanie wszystkich kolumn kategorycznych
cat_cols = df.select_dtypes(include="str").columns.tolist()

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

print(f"\nKolumny po kodowaniu — wszystkie numeryczne: {df.dtypes.nunique() == 1}")

Parental_Involvement: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}
Access_to_Resources: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}
Extracurricular_Activities: {'No': np.int64(0), 'Yes': np.int64(1)}
Motivation_Level: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}
Internet_Access: {'No': np.int64(0), 'Yes': np.int64(1)}
Family_Income: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}
Teacher_Quality: {'High': np.int64(0), 'Low': np.int64(1), 'Medium': np.int64(2)}
School_Type: {'Private': np.int64(0), 'Public': np.int64(1)}
Peer_Influence: {'Negative': np.int64(0), 'Neutral': np.int64(1), 'Positive': np.int64(2)}
Learning_Disabilities: {'No': np.int64(0), 'Yes': np.int64(1)}
Parental_Education_Level: {'College': np.int64(0), 'High School': np.int64(1), 'Postgraduate': np.int64(2)}
Distance_from_Home: {'Far': np.int64(0), 'Moderate': np.int64(1), 'Near': np.int64(2)}
Gender: {'Female': np.int64(0), 'Male': np.int6

## 4. Podział na zbiór treningowy i testowy

In [11]:
# Cechy wejściowe i zmienna docelowa
X = df.drop(columns=["Pass"])
y = df["Pass"]

# Podział 80/20, stratify zachowuje proporcje klas
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}  |  X_test: {X_test.shape}")
print(f"y_train — Pass: {y_train.sum()}  Fail: {(y_train == 0).sum()}")
print(f"y_test  — Pass: {y_test.sum()}   Fail: {(y_test == 0).sum()}")
print(f"\nCechy ({len(X.columns)}): {X.columns.tolist()}")

X_train: (5285, 19)  |  X_test: (1322, 19)
y_train — Pass: 2980  Fail: 2305
y_test  — Pass: 745   Fail: 577

Cechy (19): ['Hours_Studied', 'Attendance', 'Parental_Involvement', 'Access_to_Resources', 'Extracurricular_Activities', 'Sleep_Hours', 'Previous_Scores', 'Motivation_Level', 'Internet_Access', 'Tutoring_Sessions', 'Family_Income', 'Teacher_Quality', 'School_Type', 'Peer_Influence', 'Physical_Activity', 'Learning_Disabilities', 'Parental_Education_Level', 'Distance_from_Home', 'Gender']


## 5. Zapis danych do pliku

In [12]:
# Zapis podziału train/test — używany w kolejnych notebookach
data_splits = {
    "X_train": X_train,
    "X_test":  X_test,
    "y_train": y_train,
    "y_test":  y_test,
    "feature_names": X.columns.tolist(),
    "label_encoders": label_encoders,
}

with open("../data/splits.pkl", "wb") as f:
    pickle.dump(data_splits, f)

print("Zapisano: data/splits.pkl")

Zapisano: data/splits.pkl
